# Day 42: Human-in-the-Loop (HITL) with Chainlit and LangGraph

Welcome to Day 42 of the AI Engineering Mastery curriculum. Today we are focusing on a critical aspect of production AI systems: **Human-in-the-Loop (HITL)**. 

When deploying autonomous agents that can take actions (like deleting records, sending emails, or triggering financial transactions), you cannot blindly trust the LLM. You need a mechanism to pause execution, request human approval, and then resume.

## Core Theory (Just-in-Time)

### The "Why"
Autonomous AI agents can hallucinate or misinterpret user intent. If an agent is hooked up to a critical API, a mistake can be disastrous. Implementing HITL ensures:
1.  **Safety & Security:** Sensitive actions require explicit human sign-off.
2.  **Quality Control:** Humans can correct the agent's course if it goes off track.
3.  **Compliance:** Many enterprise use cases legally require human oversight for automated decisions.

### The "How"
We will implement HITL using **LangGraph** (for the agent state machine) and **Chainlit** (for the user interface).
1.  **LangGraph State & Breakpoints:** LangGraph allows you to define "breakpoints" before or after specific nodes in your graph. When the graph execution hits a breakpoint, it pauses and waits for external input.
2.  **Chainlit Actions:** Chainlit provides UI elements (like buttons) that allow users to interact with the chat. We can use a Chainlit Action to capture the human's approval or rejection.
3.  **Resuming State:** Once the human provides input, we update the LangGraph state and resume execution.

## Common Pitfalls in Production
1.  **State Mismatch:** If the external UI (Chainlit) and the agent state (LangGraph) get out of sync, the system can hang indefinitely waiting for input that was already provided.
2.  **Timeout Handling:** In production, humans take time to respond. Your underlying infrastructure must handle long-running state pauses gracefully (e.g., using persistent state storage rather than in-memory).
3.  **Vague Approval Prompts:** If the user is asked to approve an action but isn't given the full context (what exactly is the agent about to do?), the human approval becomes a rubber stamp, defeating the purpose.

## Setup and Dependencies

Let's install the necessary packages: `langgraph`, `chainlit`, and `langchain`.

In [1]:
# Run this in your terminal if you haven't already:
# uv pip install langgraph chainlit langchain

## 1. LangGraph HITL Architecture

First, we will build a simplified LangGraph agent that has two nodes: `reasoning_node` and `action_node`. We want to pause *before* the `action_node` to get human approval.

*Note: Since Chainlit is designed to run as a standalone server, we will simulate the graph execution and state management here in standard Python to understand the mechanics, and then show how it integrates.*

In [2]:
from typing import TypedDict, Annotated, Sequence, Any
import operator
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# 1. Define the State
class AgentState(TypedDict):
    messages: Annotated[Sequence[str], operator.add]
    approved: bool
    action_to_take: str

# 2. Define the Nodes
def reasoning_node(state: AgentState) -> dict:
    """Simulates the LLM reasoning about what action to take."""
    print("Agent is reasoning...")
    # In a real app, the LLM would decide this based on the user prompt
    action = "DELETE_DATABASE" 
    return {"messages": ["Reasoning complete."], "action_to_take": action}

def action_node(state: AgentState) -> dict:
    """Executes the action, but ONLY if approved."""
    if not state.get("approved"):
         print("Action was NOT approved. Aborting.")
         return {"messages": ["Action aborted by user."]}
    
    print(f"Executing sensitive action: {state['action_to_take']}")
    return {"messages": [f"Action {state['action_to_take']} executed successfully."]}

# 3. Build the Graph
workflow = StateGraph(AgentState)
workflow.add_node("reasoner", reasoning_node)
workflow.add_node("actor", action_node)

workflow.set_entry_point("reasoner")
workflow.add_edge("reasoner", "actor")
workflow.add_edge("actor", END)

# 4. Set the Breakpoint and Compile
# We need a checkpointer to save state when the graph pauses
memory = MemorySaver()

# We set an interrupt BEFORE the 'actor' node
app = workflow.compile(
    checkpointer=memory,
    interrupt_before=["actor"]
)

# 5. Simulate the Execution (The "Loop")
def run_simulation():
    thread_config = {"configurable": {"thread_id": "1"}}
    
    # Start execution
    print("--- Initial Run ---")
    for event in app.stream({"messages": ["Please optimize the DB"], "approved": False}, thread_config):
        print(event)
    
    # Check if we are paused
    state = app.get_state(thread_config)
    print(f"\nGraph is currently paused: {len(state.next) > 0}. Next node to run: {state.next}")
    
    if "actor" in state.next:
        # Simulate Human Input (This is what Chainlit would capture)
        print("\n--- Human Input Required ---")
        print(f"Agent wants to perform: {state.values.get('action_to_take')}")
        user_input = 'y' # Simulate user input
        
        is_approved = user_input.lower() == 'y'
        
        # Update the state with the human's decision
        app.update_state(thread_config, {"approved": is_approved})
        
        # Resume execution
        print("\n--- Resuming Run ---")
        for event in app.stream(None, thread_config):
             print(event)

run_simulation()

--- Initial Run ---
Agent is reasoning...
{'reasoner': {'messages': ['Reasoning complete.'], 'action_to_take': 'DELETE_DATABASE'}}
{'__interrupt__': ()}

Graph is currently paused: True. Next node to run: ('actor',)

--- Human Input Required ---
Agent wants to perform: DELETE_DATABASE

--- Resuming Run ---
Executing sensitive action: DELETE_DATABASE
{'actor': {'messages': ['Action DELETE_DATABASE executed successfully.']}}


## 2. Integrating with Chainlit (Concept)

To move this into a real application, you would wrap this logic in a Chainlit application (`app.py`). Here is the conceptual structure of how Chainlit maps to the LangGraph breakpoint:

```python
import chainlit as cl
# ... (LangGraph setup from above) ...

@cl.on_message
async def main(message: cl.Message):
    # 1. Start or resume the graph
    thread_config = {"configurable": {"thread_id": cl.user_session.get("id")}}
    
    # Stream the graph
    for event in app.stream({"messages": [message.content]}, thread_config):
         # Send intermediate steps to the user
         await cl.Message(content=str(event)).send()
         
    # 2. Check if we hit a breakpoint
    state = app.get_state(thread_config)
    if "actor" in state.next:
        # 3. Create a Chainlit Action (Button) for approval
        res = await cl.AskActionMessage(
            content=f"The agent wants to perform: {state.values.get('action_to_take')}. Do you approve?",
            actions=[
                cl.Action(name="approve", value="yes", label="Approve ✅"),
                cl.Action(name="reject", value="no", label="Reject ❌")
            ]
        ).send()
        
        if res and res.get("value") == "yes":
             # 4. Update state and RESUME
             app.update_state(thread_config, {"approved": True})
             # Resume graph by passing None
             for event in app.stream(None, thread_config):
                 await cl.Message(content=str(event)).send()
        else:
             app.update_state(thread_config, {"approved": False})
             for event in app.stream(None, thread_config):
                 await cl.Message(content=str(event)).send()
```

## 3. Practical Lab / Homework

**Task:**
You need to extend the provided standard Python LangGraph simulation to handle a more complex HITL scenario: **Correction**.

Instead of just `Approve` or `Reject`, the user should be able to provide feedback that forces the agent to re-reason.

1.  Modify the `AgentState` to include a `feedback` string field.
2.  Add a `human_review_node` that the graph routes to instead of interrupting before the `actor` node directly. This node is where the breakpoint will actually be. The conditional edge will be placed *after* this `human_review_node`.
3.  Add a conditional edge after `human_review_node`:
    *   If `approved` is true, go to `actor`.
    *   If `approved` is false AND `feedback` is provided, route back to the `reasoner` to try again.
4.  Update the `reasoning_node` to look at the `feedback` (if any) and change its intended action (e.g., from "DELETE_DATABASE" to "BACKUP_DATABASE").
5.  Run the simulation, reject the first action, provide feedback, and verify it loops back correctly.

In [3]:
from typing import Literal

# 1. Update State
class LabAgentState(TypedDict):
    messages: Annotated[Sequence[str], operator.add]
    approved: bool
    feedback: str
    action_to_take: str

# 2. Update Nodes
def lab_reasoning_node(state: LabAgentState) -> dict:
    print("Agent is reasoning...")
    # If there is feedback, change the action
    if state.get("feedback"):
        print(f"Agent received feedback: {state['feedback']}")
        action = "BACKUP_DATABASE"
    else:
        action = "DELETE_DATABASE"
    return {"messages": ["Reasoning complete."], "action_to_take": action}

def human_review_node(state: LabAgentState) -> dict:
    """This node just serves as a placeholder for the human in the loop breakpoint"""
    pass

def lab_action_node(state: LabAgentState) -> dict:
    if not state.get("approved"):
         print("Action was NOT approved. Aborting.")
         return {"messages": ["Action aborted by user."]}
    
    print(f"Executing sensitive action: {state['action_to_take']}")
    return {"messages": [f"Action {state['action_to_take']} executed successfully."]}

# 3. Define Conditional Routing
def route_after_human(state: LabAgentState) -> Literal["actor", "reasoner"]:
    if state.get("approved"):
        return "actor"
    elif state.get("feedback"):
        return "reasoner"
    return "actor" # Default fallback

# 4. Build Graph
lab_workflow = StateGraph(LabAgentState)
lab_workflow.add_node("reasoner", lab_reasoning_node)
lab_workflow.add_node("human_review", human_review_node)
lab_workflow.add_node("actor", lab_action_node)

lab_workflow.set_entry_point("reasoner")
lab_workflow.add_edge("reasoner", "human_review")
lab_workflow.add_conditional_edges(
    "human_review",
    route_after_human,
    {"actor": "actor", "reasoner": "reasoner"}
)
lab_workflow.add_edge("actor", END)

# Set breakpoint BEFORE human_review
lab_memory = MemorySaver()
lab_app = lab_workflow.compile(
    checkpointer=lab_memory,
    interrupt_before=["human_review"]
)

# 5. Simulation Loop
def run_lab_simulation():
    thread_config = {"configurable": {"thread_id": "lab_1"}}
    
    print("\n--- Lab Initial Run ---")
    for event in lab_app.stream({"messages": ["Please optimize the DB"], "approved": False}, thread_config):
        print(event)
        
    state = lab_app.get_state(thread_config)
    print(f"Graph paused: {len(state.next) > 0}. Next node: {state.next}")
    
    if "human_review" in state.next:
        print("\n--- Human Input Required (Correction) ---")
        print(f"Agent wants to perform: {state.values.get('action_to_take')}")
        
        # Simulate REJECTING and providing feedback
        is_approved = False
        feedback = "No, don't delete. Do a backup instead."
        print(f"User approved: {is_approved}, Feedback: '{feedback}'")
        
        # Update the state with the human's decision and feedback
        lab_app.update_state(thread_config, {"approved": is_approved, "feedback": feedback})
        
        # Resume execution
        print("\n--- Resuming Run ---")
        for event in lab_app.stream(None, thread_config):
             print(event)
             
        # Check if we hit the breakpoint again (we should!)
        state2 = lab_app.get_state(thread_config)
        if "human_review" in state2.next:
            print("\n--- Human Input Required (2nd Time) ---")
            print(f"Agent wants to perform: {state2.values.get('action_to_take')}")
            
            # Simulate APPROVING the second time
            print("User approved: True")
            lab_app.update_state(thread_config, {"approved": True, "feedback": ""})
            
            print("\n--- Resuming Run (Final) ---")
            for event in lab_app.stream(None, thread_config):
                 print(event)

run_lab_simulation()
print("\nLab graph simulation complete.")



--- Lab Initial Run ---
Agent is reasoning...
{'reasoner': {'messages': ['Reasoning complete.'], 'action_to_take': 'DELETE_DATABASE'}}
{'__interrupt__': ()}
Graph paused: True. Next node: ('human_review',)

--- Human Input Required (Correction) ---
Agent wants to perform: DELETE_DATABASE
User approved: False, Feedback: 'No, don't delete. Do a backup instead.'

--- Resuming Run ---
{'human_review': None}
Agent is reasoning...
Agent received feedback: No, don't delete. Do a backup instead.
{'reasoner': {'messages': ['Reasoning complete.'], 'action_to_take': 'BACKUP_DATABASE'}}
{'__interrupt__': ()}

--- Human Input Required (2nd Time) ---
Agent wants to perform: BACKUP_DATABASE
User approved: True

--- Resuming Run (Final) ---
{'human_review': None}
Executing sensitive action: BACKUP_DATABASE
{'actor': {'messages': ['Action BACKUP_DATABASE executed successfully.']}}

Lab graph simulation complete.
